# TxGemma Endpoint on SageMaker

이 노트북에서는 SageMaker Real-time Inference Endpoint를 사용하여 Google의 [TxGemma](https://huggingface.co/google/txgemma-2b-predict) 모델을 배포하고 약물 특성을 예측하는 방법을 실습합니다.

## TxGemma란?

TxGemma는 Google이 개발한 치료제 개발(Therapeutics) 특화 언어 모델입니다. 약물의 BBB(Blood-Brain Barrier) 투과성, 독성, 용해도 등 다양한 약물 특성을 SMILES 문자열 기반으로 예측할 수 있습니다.

## 실습 순서

1. 환경 설정
2. 모델 배포
3. 추론 테스트
4. 약물 특성 예측 (BBB 투과성)
5. 리소스 정리

## 1. 환경 설정

In [ ]:
import json

import boto3
import sagemaker
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri

# SageMaker 세션 및 역할
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

print(f"Region: {region}")
print(f"Role: {role}")

## 2. 모델 배포

Hugging Face TGI(Text Generation Inference) 컨테이너를 사용하여 TxGemma-2B-Predict 모델을 배포합니다.

### 인스턴스 권장 사양

| 인스턴스 | GPU | GPU 메모리 | 비고 |
|----------|-----|-----------|------|
| ml.g5.2xlarge | 1x A10G | 24GB | 2B 모델 권장 |
| ml.g5.12xlarge | 4x A10G | 96GB | 9B/27B 모델 |

> **참고**: TxGemma-2B-Predict는 Gated 모델이므로 Hugging Face 토큰이 필요합니다.  
> [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)에서 토큰을 발급받고,  
> [google/txgemma-2b-predict](https://huggingface.co/google/txgemma-2b-predict) 모델 페이지에서 라이선스에 동의하세요.

In [ ]:
# Hugging Face 토큰 설정
HF_TOKEN = "<REPLACE WITH YOUR TOKEN>"

assert HF_TOKEN != "<REPLACE WITH YOUR TOKEN>", \
    "Hugging Face 토큰을 입력하세요. https://huggingface.co/settings/tokens 에서 발급받을 수 있습니다."

In [ ]:
# 모델 설정
MODEL_ID = "google/txgemma-2b-predict"
INSTANCE_TYPE = "ml.g5.2xlarge"

hub = {
    "HF_MODEL_ID": MODEL_ID,
    "SM_NUM_GPUS": json.dumps(1),
    "HF_TOKEN": HF_TOKEN,
}

# TGI 컨테이너 이미지 URI
image_uri = get_huggingface_llm_image_uri("huggingface", version="3.3.6")
print(f"Model: {MODEL_ID}")
print(f"Instance: {INSTANCE_TYPE}")
print(f"Image URI: {image_uri}")

In [ ]:
# HuggingFace Model 생성 및 배포
huggingface_model = HuggingFaceModel(
    image_uri=image_uri,
    env=hub,
    role=role,
)

predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    container_startup_health_check_timeout=300,
)

print(f"Endpoint name: {predictor.endpoint_name}")

## 3. 추론 테스트

엔드포인트가 정상적으로 배포되었는지 간단한 텍스트 생성으로 확인합니다.

In [ ]:
# 기본 텍스트 생성 테스트
result = predictor.predict({
    "inputs": "My name is Julien and I like to",
})

print(result)

## 4. 약물 특성 예측 (BBB 투과성)

TxGemma를 사용하여 약물의 혈액-뇌 장벽(Blood-Brain Barrier, BBB) 투과성을 예측합니다.

### BBB란?

혈액-뇌 장벽(BBB)은 혈액과 뇌 세포외액 사이의 막으로, 대부분의 외부 약물을 차단하는 보호층입니다. 따라서 중추신경계(CNS) 약물 개발에서 BBB 투과 능력은 핵심 과제입니다.

### SMILES 표기법

SMILES(Simplified Molecular Input Line Entry System)는 분자 구조를 문자열로 표현하는 표기법입니다.

예시: `CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21`

In [ ]:
# BBB 투과성 예측
prompt = """Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21
Answer:"""

result = predictor.predict({"inputs": prompt})
print(result)

In [ ]:
# 여러 약물에 대한 BBB 투과성 배치 예측
smiles_list = [
    ("CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21", "Chlordiazepoxide (항불안제)"),
    ("CC(=O)Oc1ccccc1C(=O)O", "Aspirin (해열진통제)"),
    ("CC12CCC3c4ccc(O)cc4CCC3C1CCC2O", "Estradiol (호르몬)"),
]

prompt_template = """Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: {smiles}
Answer:"""

for smiles, name in smiles_list:
    result = predictor.predict({"inputs": prompt_template.format(smiles=smiles)})
    print(f"{name}")
    print(f"  SMILES: {smiles}")
    print(f"  Result: {result}")
    print()

## 5. 리소스 정리

> **중요**: 실습이 끝나면 반드시 엔드포인트를 삭제하여 불필요한 비용이 발생하지 않도록 합니다.

In [ ]:
# 엔드포인트 삭제
predictor.delete_endpoint()
print("Endpoint deleted.")

## 정리

이 노트북에서는 다음을 학습했습니다:

- SageMaker Real-time Endpoint를 사용한 TxGemma 모델 배포
- Hugging Face TGI 컨테이너를 활용한 LLM 서빙
- SMILES 기반 약물 특성(BBB 투과성) 예측

### 추가 실습 아이디어

- `google/txgemma-9b-predict` 또는 `google/txgemma-27b-predict`로 더 큰 모델 배포
- 약물 용해도(Solubility), 독성(Toxicity) 등 다른 특성 예측
- 여러 약물 후보에 대한 배치 예측 파이프라인 구성